# LLM Benchmarking Results Analysis

This notebook provides tools for analyzing and visualizing results from your LLM benchmarking experiments.

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Analysis environment ready!")

## 1. Load Experiment Results

In [ ]:
def load_experiment_results(results_dir='../results'):
    """
    Load all experiment result files from the results directory
    """
    results_path = Path(results_dir)
    
    # Find all JSON files (excluding summary files)
    result_files = [f for f in results_path.glob('*.json') if 'summary' not in f.name]
    
    results = []
    for file in result_files:
        with open(file, 'r') as f:
            data = json.load(f)
            results.append(data)
    
    print(f"Loaded {len(results)} experiment results")
    return results

# Load results
results = load_experiment_results()

# Display first result structure
if results:
    print("\nExample result structure:")
    print(json.dumps({k: v for k, v in results[0].items() if k != 'predictions'}, indent=2))

## 2. Create Results DataFrame

In [ ]:
def results_to_dataframe(results):
    """
    Convert results list to a pandas DataFrame
    """
    data = []
    
    for result in results:
        if result['status'] == 'completed':
            metrics = result.get('metrics', {})
            row = {
                'model': result['model'],
                'dataset': result['dataset'],
                'task_type': result['task_type'],
                'num_samples': result.get('num_samples', 0),
                'status': result['status'],
            }
            
            # Add all metrics
            for key, value in metrics.items():
                if isinstance(value, (int, float)):
                    row[f'metric_{key}'] = value
            
            data.append(row)
    
    return pd.DataFrame(data)

# Create DataFrame
df = results_to_dataframe(results)

print(f"\nDataFrame shape: {df.shape}")
display(df.head())

## 3. Summary Statistics

In [ ]:
# Group by model
print("=" * 60)
print("RESULTS BY MODEL")
print("=" * 60)

metric_cols = [col for col in df.columns if col.startswith('metric_')]

if metric_cols:
    model_summary = df.groupby('model')[metric_cols].agg(['mean', 'std', 'min', 'max'])
    display(model_summary)
else:
    print("No metrics found in results")

In [ ]:
# Group by dataset
print("=" * 60)
print("RESULTS BY DATASET")
print("=" * 60)

if metric_cols:
    dataset_summary = df.groupby('dataset')[metric_cols].agg(['mean', 'std', 'min', 'max'])
    display(dataset_summary)

## 4. Visualizations

In [ ]:
# Model comparison - Accuracy
if 'metric_accuracy' in df.columns:
    plt.figure(figsize=(12, 6))
    
    # Bar plot
    model_acc = df.groupby('model')['metric_accuracy'].mean().sort_values(ascending=False)
    
    plt.subplot(1, 2, 1)
    model_acc.plot(kind='bar', color='skyblue')
    plt.title('Average Accuracy by Model', fontsize=14, fontweight='bold')
    plt.xlabel('Model')
    plt.ylabel('Accuracy')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1)
    plt.grid(axis='y', alpha=0.3)
    
    # Box plot
    plt.subplot(1, 2, 2)
    df.boxplot(column='metric_accuracy', by='model', ax=plt.gca())
    plt.title('Accuracy Distribution by Model', fontsize=14, fontweight='bold')
    plt.suptitle('')  # Remove default title
    plt.xlabel('Model')
    plt.ylabel('Accuracy')
    plt.xticks(rotation=45, ha='right')
    plt.ylim(0, 1)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Heatmap of model performance across datasets
if 'metric_accuracy' in df.columns and len(df) > 0:
    plt.figure(figsize=(12, 8))
    
    # Create pivot table
    pivot = df.pivot_table(
        values='metric_accuracy',
        index='model',
        columns='dataset',
        aggfunc='mean'
    )
    
    # Create heatmap
    sns.heatmap(
        pivot,
        annot=True,
        fmt='.3f',
        cmap='RdYlGn',
        cbar_kws={'label': 'Accuracy'},
        vmin=0,
        vmax=1
    )
    
    plt.title('Model Performance Across Datasets (Accuracy)', fontsize=14, fontweight='bold')
    plt.xlabel('Dataset')
    plt.ylabel('Model')
    plt.tight_layout()
    plt.show()

In [ ]:
# Compare multiple metrics
if len(metric_cols) > 1:
    fig, axes = plt.subplots(1, len(metric_cols), figsize=(6*len(metric_cols), 6))
    
    if len(metric_cols) == 1:
        axes = [axes]
    
    for idx, metric_col in enumerate(metric_cols):
        metric_name = metric_col.replace('metric_', '').title()
        
        data = df.groupby('model')[metric_col].mean().sort_values(ascending=False)
        
        axes[idx].bar(range(len(data)), data.values, color='coral')
        axes[idx].set_xticks(range(len(data)))
        axes[idx].set_xticklabels(data.index, rotation=45, ha='right')
        axes[idx].set_title(f'{metric_name}', fontsize=12, fontweight='bold')
        axes[idx].set_ylabel('Score')
        axes[idx].grid(axis='y', alpha=0.3)
    
    plt.tight_layout()
    plt.show()

## 5. Error Analysis

In [ ]:
def analyze_predictions(result):
    """
    Analyze predictions from a single experiment
    """
    if 'predictions' not in result:
        return None
    
    predictions = result['predictions']
    
    correct = []
    incorrect = []
    
    for pred in predictions:
        if 'error' in pred:
            continue
        
        pred_label = str(pred['prediction']).strip().lower()
        true_label = str(pred['true_label']).strip().lower()
        
        if pred_label == true_label:
            correct.append(pred)
        else:
            incorrect.append(pred)
    
    return {
        'correct': correct,
        'incorrect': incorrect,
        'accuracy': len(correct) / (len(correct) + len(incorrect)) if (len(correct) + len(incorrect)) > 0 else 0
    }

# Analyze first result
if results:
    analysis = analyze_predictions(results[0])
    
    if analysis:
        print(f"Model: {results[0]['model']}")
        print(f"Dataset: {results[0]['dataset']}")
        print(f"Accuracy: {analysis['accuracy']:.3f}")
        print(f"Correct: {len(analysis['correct'])}")
        print(f"Incorrect: {len(analysis['incorrect'])}")
        
        # Show some incorrect predictions
        print("\nSample Incorrect Predictions:")
        for i, pred in enumerate(analysis['incorrect'][:5]):
            print(f"\n{i+1}. Input: {pred.get('input', 'N/A')[:100]}...")
            print(f"   Predicted: {pred.get('prediction')}")
            print(f"   True: {pred.get('true_label')}")

## 6. Export Results

In [ ]:
# Export to CSV
output_path = '../results/analysis_summary.csv'
df.to_csv(output_path, index=False)
print(f"Results exported to {output_path}")

# Export summary statistics
if metric_cols:
    summary_path = '../results/model_summary.csv'
    model_summary = df.groupby('model')[metric_cols].agg(['mean', 'std'])
    model_summary.to_csv(summary_path)
    print(f"Model summary exported to {summary_path}")

## 7. Custom Analysis

Add your own analysis code below:

In [ ]:
# Your custom analysis here